# Chat Skill Integration - Exemplo de Uso

Este notebook demonstra como usar o módulo `chat_skill_integration` para integrar solicitações de chat com a modificação de skills.

## 1. Importação e Inicialização

In [ ]:
# Importar o módulo principal
from chat_skill_integration import ChatSkillIntegration, Skill, SkillCategory

# Criar instância da integração
integration = ChatSkillIntegration(verbose=True)

print("Chat Skill Integration inicializado com sucesso!")

## 2. Comandos Básicos de Chat

Você pode enviar mensagens em linguagem natural para modificar skills.

In [ ]:
# Listar todas as skills disponíveis
response = integration.process_message("Listar skills")
print(response)

In [ ]:
# Adicionar uma nova skill
response = integration.process_message("Adicionar skill de previsão de tendências")
print(response)

In [ ]:
# Obter informações sobre uma skill
response = integration.process_message("Info sobre skill análise de dados")
print(response)

In [ ]:
# Desativar uma skill
response = integration.process_message("Desativar skill previsão de tendências")
print(response)

In [ ]:
# Ativar a skill novamente
response = integration.process_message("Ativar skill previsão de tendências")
print(response)

In [ ]:
# Definir parâmetro de uma skill
response = integration.process_message("Definir parâmetro threshold para 0.8 na skill análise de dados")
print(response)

In [ ]:
# Remover uma skill
response = integration.process_message("Remover skill previsão de tendências")
print(response)

## 3. Ajuda e Comandos Disponíveis

In [ ]:
# Pedir ajuda
response = integration.process_message("ajuda")
print(response)

## 4. Criando Skills com Handlers Customizados

Você pode criar skills com funções que são executadas quando a skill é chamada.

In [ ]:
# Definir uma função handler para análise customizada
def custom_analysis_handler(data=None, **kwargs):
    """Handler para análise customizada."""
    if data is None:
        return {"status": "Nenhum dado fornecido", "result": None}
    
    # Exemplo de análise simples
    result = {
        "status": "Análise concluída",
        "n_records": len(data) if hasattr(data, '__len__') else 1,
        "parameters": kwargs
    }
    return result

# Criar uma nova skill com handler
custom_skill = Skill(
    id='custom_analysis',
    name='Análise Customizada',
    description='Executa análise customizada nos dados',
    category=SkillCategory.DATA_ANALYSIS,
    parameters={'mode': 'advanced', 'threshold': 0.5}
)

# Adicionar a skill
integration.add_skill(custom_skill)

# Registrar o handler
integration.register_skill_handler('custom_analysis', custom_analysis_handler)

print("Skill customizada criada com sucesso!")

In [ ]:
# Verificar a skill criada
response = integration.process_message("Info sobre skill Análise Customizada")
print(response)

## 5. Processamento em Lote

Você pode processar múltiplas mensagens de uma vez.

In [ ]:
# Processar múltiplas mensagens
messages = [
    "Adicionar skill de detecção de anomalias",
    "Listar skills",
    "Desativar skill detecção de anomalias",
    "Listar skills"
]

responses = integration.batch_process(messages)

for msg, resp in zip(messages, responses):
    print(f"\n📝 Comando: {msg}")
    print(f"📌 Resposta: {resp}")
    print("-" * 50)

## 6. Integração com DataFrames do Pandas

Exemplo de como integrar as skills com análise de dados real.

In [ ]:
import pandas as pd
import numpy as np

# Criar um DataFrame de exemplo
np.random.seed(42)
df = pd.DataFrame({
    'idade': np.random.randint(18, 80, 100),
    'renda': np.random.normal(3000, 1500, 100),
    'categoria': np.random.choice(['A', 'B', 'C'], 100)
})

# Definir handler que usa o DataFrame
def pandas_analysis_handler(df=None, column=None, **kwargs):
    """Handler que analisa um DataFrame."""
    if df is None:
        return {"error": "DataFrame não fornecido"}
    
    result = {
        "n_rows": len(df),
        "n_columns": len(df.columns),
        "columns": list(df.columns),
        "dtypes": df.dtypes.to_dict(),
    }
    
    if column and column in df.columns:
        result[f"{column}_stats"] = df[column].describe().to_dict()
    
    return result

# Criar skill para análise de DataFrame
df_skill = Skill(
    id='dataframe_analysis',
    name='Análise de DataFrame',
    description='Analisa estrutura e estatísticas de DataFrames',
    category=SkillCategory.DATA_ANALYSIS
)

integration.add_skill(df_skill)
integration.register_skill_handler('dataframe_analysis', pandas_analysis_handler)

print("Skill de análise de DataFrame criada!")
print(f"\nDataFrame de exemplo:\n{df.head()}")

## 7. Contexto de Conversa

O sistema mantém o contexto da conversa para referências como "essa skill", "a mesma", etc.

In [ ]:
# Obter info de uma skill
response = integration.process_message("Info sobre skill limpeza de dados")
print(response)
print("\n" + "="*50 + "\n")

# Referência contextual - "essa skill" se refere à última mencionada
response = integration.process_message("Desativar essa skill")
print(response)

In [ ]:
# Ver histórico da conversa
context = integration.get_context()
print(f"Última skill referenciada: {context.last_skill_id}")
print(f"\nHistórico de interações: {len(context.history)} mensagens")

## 8. Exportar e Importar Skills

In [ ]:
# Exportar skills para arquivo
success = integration.export_skills('skills_backup.json')
print(f"Export realizado: {success}")

In [ ]:
# Ver conteúdo do arquivo exportado
import json

with open('skills_backup.json', 'r') as f:
    data = json.load(f)
    
print(json.dumps(data, indent=2, ensure_ascii=False)[:2000] + "...")

## 9. Modo Interativo

Para usar o modo interativo no terminal, execute:

```python
integration.interactive_mode()
```

Isso iniciará uma sessão de chat onde você pode digitar comandos diretamente.

## 10. Resumo dos Comandos Suportados

| Comando | Exemplos |
|---------|----------|
| Adicionar skill | "Adicionar skill de análise", "Criar skill visualização" |
| Remover skill | "Remover skill análise", "Deletar skill processamento" |
| Modificar skill | "Modificar skill análise", "Alterar skill visualização" |
| Ativar skill | "Ativar skill análise", "Habilitar skill processamento" |
| Desativar skill | "Desativar skill análise", "Desabilitar skill processamento" |
| Listar skills | "Listar skills", "Mostrar todas as habilidades" |
| Info da skill | "Info sobre skill análise", "Detalhes da skill visualização" |
| Executar skill | "Executar skill análise", "Rodar skill processamento" |
| Parâmetros | "Definir parâmetro threshold para 0.5 na skill análise" |
| Ajuda | "ajuda", "help" |

In [ ]:
# Limpar arquivos temporários (opcional)
import os
if os.path.exists('skills_backup.json'):
    os.remove('skills_backup.json')
    print("Arquivo temporário removido.")